##Live Lesson Notebook: Week 10 - Agents & Chains

In this notebook we will play with agents and chains implemented through LangChain. Note that you will not use a GPU for this notebook, a CPU is sufficient.

We will first do the usual installs and imports:

In [1]:
%%capture

!pip -q install langchain
!pip install --upgrade --quiet  langchain-community
!pip install langchainhub
!pip install -q langchain_openai
!pip install pydantic
!pip install -U duckduckgo_search

In [2]:
import torch
import os
import pprint

from langchain.llms import HuggingFacePipeline
from langchain import PromptTemplate, LLMChain
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import CharacterTextSplitter
from langchain_core.output_parsers import StrOutputParser
from langchain import hub
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings, OpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from operator import itemgetter

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

# Import things that are needed generically
from pydantic import BaseModel, Field
from langchain.tools import BaseTool, StructuredTool, tool
from langchain.tools import DuckDuckGoSearchRun, DuckDuckGoSearchResults

from langchain import hub
from langchain.agents import AgentExecutor, create_react_agent

from google.colab import userdata

###1. Chains and Chains within Chains

Now we will build our first chain. We will largely use OpenAI's GPT-3.5 Instruct model for this purpose, as it is substantially cheaper than GPT-4, and Agentic workflows can incur substantial LLM usage (be aware!). (We will however in the end also use GPT-4o to compare Agent behaviors.)  

 In order to use GPT, you have to use your OpenAI API key. (**Do NOT** print it out in the notebook! Keep it in the secrets on the left and import it as we do below, but do not show it in the notebook as clear text.)

In [4]:
OPEN_AI_KEY = userdata.get('OPEN_AI_KEY')

model = OpenAI(openai_api_key=OPEN_AI_KEY, model="gpt-3.5-turbo-instruct")
model_gpt_4o = ChatOpenAI(openai_api_key=OPEN_AI_KEY, model="gpt-4o-mini")

Let us create the first Chain. At a minimum, we need a prompt template and an LLM. An output parsers is also useful to have:

In [5]:
prompt1 = ChatPromptTemplate.from_template("In which city was {person} born? Give me only the city! Do not say '<person> was born in <city>', but just '<city>' ")

chain1 = prompt1 | model | StrOutputParser()

How do we run the chain? Let's select 'John Lennon' as the person.

In [7]:
output_1 = chain1.invoke({"person": "John Lennon"})
print(output_1)



Liverpool


Perfect! What if you want a second step to get the state of the city we got in step 1? And also get the output in another language? Can we 1) re-use chain 1, and 2) pass a second parameter for this next step? We can! Here it is:

In [8]:
prompt2 = ChatPromptTemplate.from_template(
    "Give me the country in which the city {city} is located. And also give me this attribute about that country: {country_attribute}"
)
chain2 = (
    {"city": chain1, "country_attribute": itemgetter("country_attribute")}
    | prompt2
    | model

)


In [9]:
output_2 = chain2.invoke({"person": "John Lennon", "country_attribute": "founding date"}
              )
print(output_2)



The country in which Liverpool is located is England. The founding date of England is generally considered to be 927 AD, when the Anglo-Saxon King Athelstan united the various kingdoms of the region into one nation. 


Great. The first Chain produced 'Augsburg', which then in turn became one of the inputs of the second Chain, which produced the country and the requested attribute.


##2. Tools & API Calls

Now we will see how we can call a DuckDuckGo Search within the Chain. This is an example of the API calls that we discussed.

First, what does the API look like?

In [10]:
search = DuckDuckGoSearchRun()

search.run("George Washington birthdate")

/usr/local/lib/python3.11/dist-packages/langchain_community/utilities/duckduckgo_search.py:63: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


'George is a good little monkey…and always very curious! For over 80 years, the adventures of George and his … George Washington, the first president of the United States George (English: / ˈdʒɔːrdʒ /) is a masculine given name … May 30, 2025 · From the Greek name Γεώργιος (Georgios), which was derived from the Greek word γεωργός … [ 1 syll. geo - (r) ge, ge -o- rge ] The baby boy name George is pronounced as JH OW RJH (English) †. George is used … George is a traditionally masculine name with Greek and English roots. The prevailing meaning of George is …'

Note that this is a search result and not an LLM answer! So you recover text which usually contains a lot more information than simply the answer to your question. (That's one reason why RAG is superior to search.)

Now we will put it in a chain. We will do this in three steps:

1) rewrite the question as a search query using the LLM.     
2) Send the query to the search tool.   
3) construct the answer using the LLM.




In [11]:
template_search_rewrite = """Turn the following user input into a search query for a search engine:\n\n
{input}"""

template_answer = """Based on this search result:\n\n
{search_result},
\n\n
think through it step by step to give an answer to this purpose: {purpose}.
End your answer with:
Final answer: <just the answer addressing the purpose, not more>
"""

prompt_search_rewrite = ChatPromptTemplate.from_template(template_search_rewrite)

prompt_answer = ChatPromptTemplate.from_template(template_answer)

search = DuckDuckGoSearchRun()


chain_search_query_rewrite = prompt_search_rewrite | model | StrOutputParser()

chain_search = chain_search_query_rewrite | search

full_chain = (
    {"search_result": chain_search, "input": itemgetter("input"), "purpose": itemgetter("purpose")}
    | prompt_answer
    | model
    | StrOutputParser()
)


As we can see, *chain_search_query* using the LLM rewrites the 'input' (a thought) into a more suitabke search query. *chain_search* inherits that chain and adds a tool use, and *full_chain* combines the search result and the 'purpose' to construct the answer using the LLM.

Let's see the outputs of all of the (sub-)chains for this situation:

* input:   *I wonder when George Washington was born.*
* purpose: *Washington's age in 1767*

Note that the *chain_search_query* and *chain_search* chains do not depend on the 'purpose', so we will not provide that argument for theior invocations.

In [12]:
chain_search_query_rewrite.invoke({"input": "I wonder when George Washington was born."})

'\n\n"George Washington birth date"'

In [13]:
chain_search.invoke({"input": "I wonder when George Washington was born."})

/usr/local/lib/python3.11/dist-packages/langchain_community/utilities/duckduckgo_search.py:63: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


'George is a good little monkey…and always very curious! For over 80 years, the adventures of George and his friend The Man With the … George Washington, the first president of the United States George (English: / ˈdʒɔːrdʒ /) is a masculine given name derived from the … May 30, 2025 · From the Greek name Γεώργιος (Georgios), which was derived from the Greek word γεωργός (georgos) meaning "farmer, … [ 1 syll. geo - (r) ge, ge -o- rge ] The baby boy name George is pronounced as JH OW RJH (English) †. George is used predominantly in … George is a traditionally masculine name with Greek and English roots. The prevailing meaning of George is "farmer" — in Greek it …'

In [14]:
print(full_chain.invoke({"input": "I wonder when George Washington was born.",
                   "purpose": "Washington's age in 1767"}))

/usr/local/lib/python3.11/dist-packages/langchain_community/utilities/duckduckgo_search.py:63: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
<frozen importlib._bootstrap>:1047: ImportWarning: _PyDriveImportHook.find_spec() not found; falling back to find_module()
<frozen importlib._bootstrap>:1047: ImportWarning: _BokehImportHook.find_spec() not found; falling back to find_module()
<frozen importlib._bootstrap>:1047: ImportWarning: _PyDriveImportHook.find_spec() not found; falling back to find_module()
<frozen importlib._bootstrap>:1047: ImportWarning: _BokehImportHook.find_spec() not found; falling back to find_module()
sys:1: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7dec9405d160>



Step 1: Determine the year of 1767.
1767 falls between the years of 1752 and 1775, during which Washington would have been 35-43 years old.

Step 2: Determine Washington's birth year.
According to the search result, Washington was born in 1732.

Step 3: Calculate the difference between 1767 and 1732.
1767-1732 = 35

Step 4: Add the difference to Washington's birth year.
1732 + 35 = 1767

Final answer: In 1767, George Washington was 35 years old.


Good. This shows the use of API calls together with LLMs.

We will now turn to agents.

##3. Agents

We will now build a simple ReAct agent (there are others!). In fact, we will build custom tools (fake - they will return static answers) and see how a simple agent query could make use of the tools to come up with the answer. We will also see how the decision process differs depending on the tools available.

To illustrate how to build a tool - here is an example:

In [15]:
@tool
def magic_answers(query: str) -> str:
    """Create a magic answer!"""
    return f'This should be a magic answer, answering this query question: {query}'

In [16]:
print(magic_answers.name)
print(magic_answers.description)
print(magic_answers.args)

magic_answers
Create a magic answer!
{'query': {'title': 'Query', 'type': 'string'}}


In [17]:
template = """Turn the following user input into a search query for a search engine:\n\n
{input}"""
prompt = ChatPromptTemplate.from_template(template)

chain_test = prompt | model | StrOutputParser() | magic_answers

In [18]:
chain_test.invoke({"input": "It may be useful to figure out what the temperatutre is in New York right now?"})

'This should be a magic answer, answering this query question: \n\n"Temperature in New York"'

Next, we will get real and apply this to agents. We will get 'the **React Prompt**' from the LangChain hub (note: no reason why you could not change it!), and then we'll define a few fake tools that may return useful info for our question that we will ask eventually:

In [19]:
# Get the prompt to use - you can modify this!
#Ignore the API key warning
react_prompt = hub.pull("hwchase17/react")

/usr/local/lib/python3.11/dist-packages/langsmith/client.py:272: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(
<frozen importlib._bootstrap>:1047: ImportWarning: _PyDriveImportHook.find_spec() not found; falling back to find_module()
<frozen importlib._bootstrap>:1047: ImportWarning: _BokehImportHook.find_spec() not found; falling back to find_module()


In [20]:
react_prompt

PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')

In [21]:
print(react_prompt.template)

Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}


Now we will build a few fake tools, most of them simply returning pre-canned answers relevant to the question we will ask later. Obviously, in reality these would be actual APIs that use inputs to return specific information. But for this illustration we are simply interested in the principle of how tools work, and how Agents use Tools.

In [22]:
@tool
def multiply(t) -> str:
    """Multiply two numbers. The input format should be: a, b"""
    a, b = t.split(",")
    return float(a) * float(b)

@tool
def my_core_reasons(activity: str) -> str:
    """Find the underlying reason for the described activity to take place"""
    return f"It happened because someone came up with the idea and we all thought it was a great idea at the time."

@tool
def my_other_data(activity: str) -> str:
    """Find additional data around the given activity"""
    return f"We were freezing and two people got sick!"

@tool
def my_temperature(date: str) -> str:
    """Find the temperature at a given date"""
    return f"It was 3.1 degrees Celsius that day."



Now let's construct three tool sets with different tool content:

In [23]:
tools_0 = [my_temperature]
tools_1 = [my_core_reasons, my_other_data, my_temperature]
tools_2 = [my_core_reasons, my_other_data, my_temperature, multiply]

Next, we'll construct three corresponding React agents and their execution environments::

In [24]:
# Construct the ReAct agents
agent_0 = create_react_agent(model, tools_0, react_prompt)
agent_1 = create_react_agent(model, tools_1, react_prompt)
agent_2 = create_react_agent(model, tools_2, react_prompt)

In [25]:
# Create an agent executor by passing in the agent and tools
agent_executor_0 = AgentExecutor(agent=agent_0, tools=tools_0, verbose=True)
agent_executor_1 = AgentExecutor(agent=agent_1, tools=tools_1, verbose=True)
agent_executor_2 = AgentExecutor(agent=agent_2, tools=tools_2, verbose=True)



Now we will execute the three agents. Which differences do you see?

In [26]:
agent_executor_0.invoke({"input": "Was it a good idea to go swimming last night?"})  # access only to temperature

<frozen importlib._bootstrap>:1047: ImportWarning: _PyDriveImportHook.find_spec() not found; falling back to find_module()
<frozen importlib._bootstrap>:1047: ImportWarning: _BokehImportHook.find_spec() not found; falling back to find_module()




> Entering new AgentExecutor chain...
 I need to find out the temperature last night
Action: my_temperature
Action Input: "last night"It was 3.1 degrees Celsius that day.3.1 degrees Celsius is too cold for swimming, so it was probably not a good idea.
Final Answer: No, it was not a good idea to go swimming last night.

> Finished chain.


{'input': 'Was it a good idea to go swimming last night?',
 'output': 'No, it was not a good idea to go swimming last night.'}

In [27]:
agent_executor_1.invoke({"input": "Was it a good idea to go swimming last night?"})      # access also to my_core_reasons and my_other_data



> Entering new AgentExecutor chain...
 I should find out additional data around swimming last night
Action: my_other_data
Action Input: "swimming"We were freezing and two people got sick! I should check the temperature for last night
Action: my_temperature
Action Input: "last night"It was 3.1 degrees Celsius that day.Based on the low temperature, I should check the underlying reason for going swimming.
Action: my_core_reasons
Action Input: "swimming"It happened because someone came up with the idea and we all thought it was a great idea at the time. I now know the final answer
Final Answer: The reason for going swimming last night was because someone suggested it and everyone agreed, even though it was too cold and two people got sick.

> Finished chain.


{'input': 'Was it a good idea to go swimming last night?',
 'output': 'The reason for going swimming last night was because someone suggested it and everyone agreed, even though it was too cold and two people got sick.'}

In [28]:
agent_executor_2.invoke({"input": "Was it a good idea to go swimming last night?"})         # access now also to multiplication tool



> Entering new AgentExecutor chain...
 The question is subjective, but let's see if we can find any underlying reasons or additional data around the activity.
Action: my_core_reasons
Action Input: swimmingIt happened because someone came up with the idea and we all thought it was a great idea at the time. This gives us some insight into the underlying reason - someone suggested it and everyone agreed it was a good idea.
Action: my_other_data
Action Input: swimmingWe were freezing and two people got sick! This additional data shows that the temperature may have played a role in the activity and that it may not have been the best decision.
Action: my_temperature
Action Input: last nightIt was 3.1 degrees Celsius that day. This confirms that the temperature was quite low and could have contributed to people getting sick.
Action: multiply
Action Input: 3.1, 26.2 Multiplying the temperature by 2 could represent the potential impact it had on the activity.
Final Answer: It may not have bee

{'input': 'Was it a good idea to go swimming last night?',
 'output': 'It may not have been the best idea to go swimming last night due to the low temperature and resulting sickness.'}

So this did not work right! Why did it even call the Multiply tool? It did a completely meaningless calculation and drew a random conclusion!

So one may want to be purposeful in selecting the Tools for an Agent.

However, let's see whether a strong model could do better. Let us repeat the exercise with GPT-4o:

In [29]:
# Construct the ReAct agents
agent_gpt_4o_0 = create_react_agent(model_gpt_4o, tools_0, react_prompt)
agent_gpt_4o_1 = create_react_agent(model_gpt_4o, tools_1, react_prompt)
agent_gpt_4o_2 = create_react_agent(model_gpt_4o, tools_2, react_prompt)

# Create an agent executor by passing in the agent and tools
agent_executor_gpt_4o_0 = AgentExecutor(agent=agent_gpt_4o_0, tools=tools_0, verbose=True)
agent_executor_gpt_4o_1 = AgentExecutor(agent=agent_gpt_4o_1, tools=tools_1, verbose=True)
agent_executor_gpt_4o_2 = AgentExecutor(agent=agent_gpt_4o_2, tools=tools_2, verbose=True)


What happens if we run the agent executor that has access to all tools?

In [30]:
agent_executor_gpt_4o_2.invoke({"input": "Was it a good idea to go swimming last night?"})



> Entering new AgentExecutor chain...
To determine if it was a good idea to go swimming last night, I should gather information on the weather, specifically the temperature, since it greatly affects swimming conditions. 

Action: my_temperature 
Action Input: "last night" It was 3.1 degrees Celsius that day.Given that the temperature was quite low at 3.1 degrees Celsius, it may not have been a good idea to go swimming last night, as such conditions could be uncomfortable and potentially unsafe for swimming.

Final Answer: It was likely not a good idea to go swimming last night due to the low temperature of 3.1 degrees Celsius.

> Finished chain.


{'input': 'Was it a good idea to go swimming last night?',
 'output': 'It was likely not a good idea to go swimming last night due to the low temperature of 3.1 degrees Celsius.'}

This seems much more reasonable! The Multiply Tool was not called. In fact, only the my_temperature tool was called, and - resonably - the model decided it had all it needed.

So it appears that, as expected, stronger models are better in deciding which tools to use and how to use the information.